In [1]:
import pandas as pd
import glob
import os
import numpy as np

In [2]:
# BAGIAN PENGGABUNGAN DATASET HARGA CABAI RAWIT

In [3]:
path = "C:\\Users\\User\\tkti_dataset\\dataset_pangan"
all_files = glob.glob(os.path.join(path, "*xlsx"))

list_df_pangan = []
print(f"Menemukan {len(all_files)} file. Memulai proses pembersihan...")

Menemukan 5 file. Memulai proses pembersihan...


In [4]:
for filename in all_files:
    try:
        # 1. Baca Excel
        df_raw = pd.read_excel(filename)
        
        # 2. Normalisasi kolom komoditas agar tidak sensitif spasi/huruf besar
        df_raw['Komoditas (Rp)'] = df_raw['Komoditas (Rp)'].astype(str).str.strip()
        
        # 3. Filter: Pakai regex agar lebih fleksibel (bisa menangkap Merah/Hijau jika mau)
        # Sesuai permintaan, fokus ke Cabai Rawit Merah
        df_filtered = df_raw[df_raw['Komoditas (Rp)'] == 'Cabai Rawit Merah'].copy()
        if df_filtered.empty:
            print(f"Peringatan: Tidak menemukan baris 'Cabai Rawit Merah' di {os.path.basename(filename)}")
            continue
            
        if 'No' in df_filtered.columns:
            df_filtered = df_filtered.drop(columns=['No'])
            
        # 4. Melt
        df_melted = df_filtered.melt(
            id_vars=['Komoditas (Rp)'], 
            var_name='Tanggal', 
            value_name='Harga'
        )
        
        list_df_pangan.append(df_melted)
        print(f"Berhasil unpivot: {os.path.basename(filename)} ({len(df_melted)} baris)")
        
    except Exception as e:
        print(f"Gagal memproses {os.path.basename(filename)}: {e}")

if list_df_pangan:
    df_pangan_final = pd.concat(list_df_pangan, axis=0, ignore_index=True)
    
    # 5. Pembersihan Tanggal yang lebih kuat
    df_pangan_final['Tanggal'] = df_pangan_final['Tanggal'].astype(str).str.replace(' ', '', regex=False)
    df_pangan_final['Tanggal'] = pd.to_datetime(df_pangan_final['Tanggal'], dayfirst=True, errors='coerce')
    
    # 6. Pembersihan Harga (Menghapus titik ribuan agar bisa jadi angka)
    # Contoh: "39.500" -> "39500"
    df_pangan_final['Harga'] = df_pangan_final['Harga'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    df_pangan_final['Harga'] = pd.to_numeric(df_pangan_final['Harga'].replace('-', np.nan), errors='coerce')
    
    # 7. Sort & Fill
    df_pangan_final = df_pangan_final.sort_values('Tanggal')
    df_pangan_final['Harga'] = df_pangan_final['Harga'].ffill()
    
    # Cek jumlah data sebelum dropna untuk diagnosa
    print(f"\nJumlah data sebelum filter NaT/NaN: {len(df_pangan_final)}")
    
    # 8. Drop baris yang benar-benar tidak bisa diperbaiki
    df_pangan_final = df_pangan_final.dropna(subset=['Tanggal', 'Harga'])
    
    df_pangan_final = df_pangan_final[['Tanggal', 'Harga']]
    df_pangan_final.to_csv('data_pangan_merge_raw.csv', index=False)
    
    print("\nDataset Pangan Berhasil Digabung!")
    print(f"Total baris data akhir: {len(df_pangan_final)}")
    if not df_pangan_final.empty:
        print(df_pangan_final.head())
else:
    print("Tidak ada data pangan yang ditemukan.")

Berhasil unpivot: Tabel Harga 2020.xlsx (262 baris)
Berhasil unpivot: Tabel Harga 2021.xlsx (261 baris)
Berhasil unpivot: Tabel Harga 2022.xlsx (260 baris)
Berhasil unpivot: Tabel Harga 2023-2024.xlsx (522 baris)
Berhasil unpivot: Tabel Harga 2025-Mar 2026.xlsx (325 baris)

Jumlah data sebelum filter NaT/NaN: 1630

Dataset Pangan Berhasil Digabung!
Total baris data akhir: 1629
     Tanggal  Harga
1 2020-01-02  39.50
2 2020-01-03  39.50
3 2020-01-06  37.75
4 2020-01-07  37.75
5 2020-01-08  37.75


In [5]:
# BAGIAN PENGGABUNGAN DATASET CURAH HUJAN

In [6]:
path = "C:\\Users\\User\\tkti_dataset"
all_files = glob.glob(os.path.join(path, "*xlsx"))

list_df = []
print(f"Menemukan {len(all_files)} file. Memulai proses pembersihan...")

Menemukan 75 file. Memulai proses pembersihan...


In [9]:
if list_df:
    # 1. Gabungkan semua file menjadi satu dataframe besar
    df_raw_concat = pd.concat(list_df, axis=0, ignore_index=True)

    # 2. Konversi Tanggal TERLEBIH DAHULU
    # Gunakan 'coerce' untuk mengubah teks "KETERANGAN:" menjadi NaT
    df_raw_concat['TANGGAL'] = pd.to_datetime(
        df_raw_concat['TANGGAL'], 
        dayfirst=True, 
        errors='coerce'
    )

    # 3. Buang baris yang bukan tanggal (footer/NaT)
    df_raw_concat = df_raw_concat.dropna(subset=['TANGGAL'])

    # 4. PROSES DEDUPLIKASI (Aggregation)
    # Kita mengelompokkan berdasarkan tanggal yang sudah seragam formatnya.
    # Jika ada dua nilai di hari yang sama, kita ambil rata-ratanya (mean).
    df_cuaca_clean = df_raw_concat.groupby('TANGGAL').agg({
        'TAVG': 'mean', 
        'RH_AVG': 'mean', 
        'RR': 'mean'
    }).reset_index()

    # 5. Urutkan secara kronologis
    df_cuaca_clean = df_cuaca_clean.sort_values('TANGGAL')

    # 6. Simpan hasil yang SUDAH BERSIH (df_cuaca_clean)
    df_cuaca_clean.to_csv('data_cuaca_merge_clean.csv', index=False)

    print("--- RINGKASAN PROSES ---")
    print(f"Total baris mentah gabungan: {len(df_raw_concat)}")
    print(f"Total baris unik (setelah deduplikasi): {len(df_cuaca_clean)}")
    print(f"Rentang Data: {df_cuaca_clean['TANGGAL'].min()} s/d {df_cuaca_clean['TANGGAL'].max()}")
    
    # Menampilkan 5 baris pertama untuk verifikasi
    print("\nPreview Data:")
    print(df_cuaca_clean.head())
else:
    print("Tidak ada data ditemukan.")

--- RINGKASAN PROSES ---
Total baris mentah gabungan: 4560
Total baris unik (setelah deduplikasi): 2280
Rentang Data: 2020-01-01 00:00:00 s/d 2026-03-31 00:00:00

Preview Data:
     TANGGAL  TAVG  RH_AVG   RR
0 2020-01-01  23.8    82.0  0.2
1 2020-01-02   NaN     NaN  0.0
2 2020-01-03  22.1    92.0  0.5
3 2020-01-04  22.2    94.0  7.6
4 2020-01-05  22.0    94.0  4.3


In [10]:
# BAGIAN PRE-PROCESSING

In [11]:
# 1. Load Data (Menggunakan file yang sudah Anda siapkan)
df_cuaca = pd.read_csv('data_cuaca_merge_clean.csv') 
df_pangan = pd.read_csv('data_pangan_merge_raw.csv')

# Pastikan tipe data Tanggal seragam
df_cuaca['Tanggal'] = pd.to_datetime(df_cuaca['TANGGAL'])
df_pangan['Tanggal'] = pd.to_datetime(df_pangan['Tanggal'])

# 2. Buat "Master Calendar" (Agar tidak ada tanggal yang terlewat dari 2020 - 2026)
start_date = df_cuaca['Tanggal'].min()
end_date = df_cuaca['Tanggal'].max()
master_calendar = pd.DataFrame({'Tanggal': pd.date_range(start=start_date, end=end_date)})

# 3. Gabungkan Semua ke Master Calendar (Outer Join)
df_final = pd.merge(master_calendar, df_cuaca.drop(columns=['TANGGAL']), on='Tanggal', how='left')
df_final = pd.merge(df_final, df_pangan, on='Tanggal', how='left')

# 4. PENANGANAN DATA KOSONG (Imputation)
# A. Data Cuaca: Gunakan Interpolasi Linear untuk mengisi 2 hari yang bolong
df_final[['TAVG', 'RH_AVG', 'RR']] = df_final[['TAVG', 'RH_AVG', 'RR']].interpolate(method='linear')

# B. Data Pangan: Gunakan Forward Fill (ffill) untuk mengisi harga di hari libur/Sabtu/Minggu
# Ini mengasumsikan harga rica di hari libur tetap mengikuti harga terakhir saat pasar buka.
df_final['Harga'] = df_final['Harga'].ffill()

# 5. FEATURE ENGINEERING (Penting untuk Akurasi ML)
# A. Tambahkan Fitur Waktu
df_final['Bulan'] = df_final['Tanggal'].dt.month
df_final['Hari_ke'] = df_final['Tanggal'].dt.dayofweek # 0=Senin, 6=Minggu
df_final['Tahun'] = df_final['Tanggal'].dt.year

# B. Fitur Hari Raya Manado (Biner) - Mengingat lonjakan harga rica biasanya di momen ini
def is_holiday_manado(row):
    # Contoh: Natal (24-25 Des) dan Tahun Baru (31 Des - 1 Jan)
    if (row.month == 12 and row.day in [24, 25, 31]) or (row.month == 1 and row.day == 1):
        return 1
    return 0

df_final['Is_Hari_Raya'] = df_final['Tanggal'].apply(is_holiday_manado)

# C. Fitur Lag (Harga 1 & 7 hari sebelumnya)
# Model ML butuh melihat tren harga kemarin dan minggu lalu
df_final['Harga_Lag_1'] = df_final['Harga'].shift(1)
df_final['Harga_Lag_7'] = df_final['Harga'].shift(7)

# 6. Bersihkan baris pertama yang NaN akibat proses Lag
df_final = df_final.dropna()

# 7. EXPORT
df_final.to_csv('clean_dataset_rica_manado_final.csv', index=False)

print(f"--- PROSES FINAL SELESAI ---")
print(f"Total baris final: {len(df_final)}")
print(f"Dataset siap digunakan untuk training Machine Learning!")

--- PROSES FINAL SELESAI ---
Total baris final: 2274
Dataset siap digunakan untuk training Machine Learning!
